# GreenWave — MAPPO/CTDE Training on Google Colab

## 계정별 설정
| 계정 | MODE | 설명 |
|------|------|------|
| Account 1 | `"MAPPO"` | Decentralized critic |
| Account 2 | `"CTDE"` | Centralized critic |

**시작 전 체크리스트**
1. 런타임 > 런타임 유형 변경 > **T4 GPU** 선택
2. `GreenWave.zip` 을 Google Drive 최상위(`내 드라이브`)에 업로드
3. 아래 `⚙️ 설정` 셀에서 `MODE` 만 변경


In [ ]:
# ══════════════════════════════════════════════════════════════
#  ⚙️  설정 — 이 셀만 바꿔서 Account 1 / Account 2 를 구분
# ══════════════════════════════════════════════════════════════

MODE             = "MAPPO"   # Account 1: "MAPPO"  |  Account 2: "CTDE"
MAP              = "2x2-brt"
NUM_ITERS        = 80
NUM_WORKERS      = 2         # Colab RAM 기준 (≥4 는 메모리 부족 위험)
TRAIN_BATCH_SIZE = 4000      # 기본 8000 → 4000 으로 iter 시간 ~2배 단축

# Google Drive 경로
PROJECT_ZIP_NAME = "GreenWave.zip"          # Drive 최상위에 업로드한 파일명
GDRIVE_SAVE_DIR  = "/content/drive/MyDrive/GreenWave_models"  # 결과 저장 위치

# ── 자동 결정 (수정 불필요) ──────────────────────────────────
import os
MODEL_TAG  = "MAPPO_CTDE" if MODE == "CTDE" else "MAPPO"
OUT_PATH   = f"{GDRIVE_SAVE_DIR}/{MODEL_TAG}_colab_{MAP.replace('-','_')}"
CTDE_FLAG  = "--ctde" if MODE == "CTDE" else ""

print(f"MODE={MODE}  MAP={MAP}  iters={NUM_ITERS}  workers={NUM_WORKERS}")
print(f"train_batch_size={TRAIN_BATCH_SIZE}  out={OUT_PATH}")


---
## Step 1 — GPU 확인 + Google Drive 마운트

In [ ]:
# GPU 확인
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print("GPU:", r.stdout.strip() or "Not found — CPU mode")

# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted ✓")


---
## Step 2 — SUMO 1.20 설치 (Ubuntu PPA)
약 1~2분 소요. 셀 출력 마지막에 `SUMO 1.20.x` 버전이 출력되면 OK.

In [ ]:
%%bash
set -e
# SUMO stable PPA 추가
sudo add-apt-repository ppa:sumo/stable -y 2>/dev/null
sudo apt-get update -qq
sudo apt-get install -y sumo sumo-tools sumo-doc 2>&1 | grep -E '(Setting up|already|error)' || true
echo "--- SUMO binary ---"
sumo --version 2>&1 | head -3
echo "--- netconvert ---"
which netconvert && netconvert --version 2>&1 | head -1 || echo "netconvert not found!"


---
## Step 3 — 프로젝트 압축 해제

**사전 준비 (로컬 Mac에서 실행):**
```bash
cd /Volumes/june2_07ssd/PycharmProjects
zip -r ~/Desktop/GreenWave.zip GreenWave/ \
  --exclude "GreenWave/.venv/*" \
  --exclude "GreenWave/models/*" \
  --exclude "GreenWave/results/*" \
  --exclude "GreenWave/videos/*" \
  --exclude "GreenWave/__pycache__/*" \
  --exclude "GreenWave/**/__pycache__/*" \
  --exclude "GreenWave/**/*.pyc"
```
생성된 `~/Desktop/GreenWave.zip` 을 Google Drive 최상위(`내 드라이브`)에 업로드.

In [ ]:
import zipfile, os, shutil

zip_src = f"/content/drive/MyDrive/{PROJECT_ZIP_NAME}"
extract_to = "/content"

if not os.path.exists(zip_src):
    raise FileNotFoundError(f"{zip_src} not found. Drive에 {PROJECT_ZIP_NAME} 업로드 확인.")

print(f"Extracting {zip_src} → {extract_to} ...", end=" ")
with zipfile.ZipFile(zip_src, 'r') as z:
    z.extractall(extract_to)
print("done")

# 압축 내용에 따라 경로가 다를 수 있음 — 자동 감지
project_dir = "/content/GreenWave"
if not os.path.isdir(project_dir):
    # zip 루트 자체가 GreenWave 파일들인 경우
    os.rename("/content", project_dir)

print("Project files:")
print([f for f in os.listdir(project_dir) if not f.startswith('.')])


---
## Step 4 — Python 의존성 설치
약 3~5분 소요.

In [ ]:
%%bash
set -e

# SUMO 버전 확인 (traci/sumolib 버전 매칭용)
SUMO_VER=$(sumo --version 2>&1 | grep -oP '(?<=Version )\S+' | head -1)
echo "Detected SUMO version: $SUMO_VER"

# 핵심 패키지
pip install -q \
  "ray[rllib]>=2.10.0" \
  torch pettingzoo gymnasium \
  numpy pandas \
  imageio imageio-ffmpeg \
  tensorboard matplotlib \
  2>&1 | tail -3

# SUMO Python 바인딩 — SUMO 바이너리 버전에 맞춰 설치
pip install -q traci==${SUMO_VER} sumolib==${SUMO_VER} 2>&1 | tail -3 || \
pip install -q traci sumolib 2>&1 | tail -3

echo ""
echo "=== 버전 확인 ==="
python -c "import ray; print('ray', ray.__version__)"
python -c "import torch; print('torch', torch.__version__)"
python -c "import traci; print('traci', traci.VERSION)"
echo "done"


---
## Step 5 — 환경 변수 설정 + SUMO smoke test

In [ ]:
import os, sys

# SUMO_HOME — Ubuntu PPA 설치 기본 경로
SUMO_HOME = "/usr/share/sumo"
os.environ['SUMO_HOME'] = SUMO_HOME
os.environ['PYTHONPATH'] = f"{SUMO_HOME}/tools:{os.environ.get('PYTHONPATH', '')}"

# 프로젝트 루트를 sys.path 에 추가 (상대 import 지원)
PROJECT_DIR = "/content/GreenWave"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

# models/, results/ 디렉터리 생성 (없으면 train 스크립트가 실패)
os.makedirs(f"{PROJECT_DIR}/models", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)

# Drive 저장 디렉터리 생성
os.makedirs(GDRIVE_SAVE_DIR, exist_ok=True)

# SUMO 접근 가능 확인
import subprocess
r = subprocess.run(['sumo', '--version'], capture_output=True, text=True)
print("SUMO:", r.stdout.split('\n')[0])

# traci import 확인
import traci
import sumolib
print(f"traci {traci.VERSION}  sumolib OK")
print(f"\n프로젝트 경로: {os.getcwd()}")
print(f"저장 경로: {OUT_PATH}")


---
## Step 6 — 학습 실행

> **예상 소요 시간 (T4 기준)**  
> `train_batch_size=4000`, `workers=2`, `iters=80` →  
> iter당 약 90~120초 × 80 = **약 2~2.7시간**
>
> - 중간에 브라우저 탭을 닫아도 런타임은 계속 실행됩니다 (Colab Pro 최대 24h).
> - 모델은 Google Drive에 직접 저장되어 연결 끊겨도 손실 없음.
> - 중단됐다면 아래 `Step 6b (이어서 학습)` 셀로 재시작 가능.

이 셀은 학습이 끝날 때까지 블록됩니다. 출력 스트림에 iter 진행 상황이 표시됩니다.

In [ ]:
import subprocess, sys, os

# 환경 변수 서브프로세스에 전달
env = os.environ.copy()
env['SUMO_HOME'] = '/usr/share/sumo'
env['PYTHONPATH'] = f"/usr/share/sumo/tools:{env.get('PYTHONPATH', '')}"

cmd = [
    sys.executable, 'train_mappo.py',
    '--map', MAP,
    '--num-iters', str(NUM_ITERS),
    '--num-workers', str(NUM_WORKERS),
    '--train-batch-size', str(TRAIN_BATCH_SIZE),
    '--out', OUT_PATH,
]
if MODE == "CTDE":
    cmd.append('--ctde')

print("실행 명령:", ' '.join(cmd))
print("="*60)

# 출력을 실시간 스트리밍 (Jupyter stdout 에 바로 표시)
proc = subprocess.Popen(
    cmd,
    cwd='/content/GreenWave',
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f"\n학습 종료 (exit code: {proc.returncode})")


---
## Step 6b — 이어서 학습 (중단됐을 때만 실행)
세션이 끊겼다면: Step 1~5 재실행 후 이 셀만 실행.

In [ ]:
# 이어서 학습할 추가 iter 수
EXTRA_ITERS = 40  # 이 값만 조정

import subprocess, sys, os
env = os.environ.copy()
env['SUMO_HOME'] = '/usr/share/sumo'
env['PYTHONPATH'] = f"/usr/share/sumo/tools:{env.get('PYTHONPATH', '')}"

cmd = [
    sys.executable, 'train_mappo.py',
    '--map', MAP,
    '--num-iters', str(EXTRA_ITERS),
    '--num-workers', str(NUM_WORKERS),
    '--train-batch-size', str(TRAIN_BATCH_SIZE),
    '--resume-from', OUT_PATH,  # Drive에서 이어서
    '--out', OUT_PATH,           # 같은 경로에 덮어쓰기
]
if MODE == "CTDE":
    cmd.append('--ctde')

print("이어서 학습:", ' '.join(cmd))
print("="*60)

proc = subprocess.Popen(
    cmd,
    cwd='/content/GreenWave',
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f"\n이어서 학습 종료 (exit code: {proc.returncode})")


---
## Step 7 — 결과 확인 + 로컬 다운로드

학습이 완료되면:
1. Google Drive에서 `GreenWave_models/MAPPO_colab_2x2_brt/` (또는 `MAPPO_CTDE_...`) 폴더 확인
2. 그 폴더 전체를 로컬에 다운로드
3. 로컬 `models/` 아래 배치 후 `--resume-from` 또는 `evaluate_mappo.py` 에서 사용

In [ ]:
import os, json

meta_path = os.path.join(OUT_PATH, 'train_metadata.json')
if os.path.exists(meta_path):
    meta = json.loads(open(meta_path).read())
    print(f"model_name   : {meta.get('model_name')}")
    print(f"train_iter   : {meta.get('train_iter')}")
    print(f"map          : {meta.get('map')}")
    print(f"ctde_mode    : {meta.get('ctde_mode')}")
    print(f"train_batch  : {meta.get('train_batch_size')}")
    print(f"num_workers  : {meta.get('num_workers')}")
    print(f"\n저장 경로 → {OUT_PATH}")
    print("Google Drive 에서 이 폴더를 다운로드해서 로컬 models/ 에 배치하세요.")
else:
    print("메타데이터 파일 없음 — 학습이 완료되지 않았거나 저장 경로 확인 필요.")
    print(f"확인 경로: {OUT_PATH}")

# 모델 파일 목록 출력
print("\n저장된 파일:")
for root, dirs, files in os.walk(OUT_PATH):
    level = root.replace(OUT_PATH, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # 2단계까지만 표시
        for f in files:
            print(f"{indent}  {f}")


---
## Step 8 — 로컬에서 모델 사용 방법

다운로드한 폴더를 로컬 `models/` 아래에 배치 후:

```bash
# 평가 (3-way 비교)
python evaluate_mappo.py \
  --model models/MAPPO_colab_2x2_brt \
  --model-ctde models/MAPPO_CTDE_colab_2x2_brt \
  --map 2x2-brt --episodes 5

# 이어서 로컬 학습 (추가 fine-tune)
python train_mappo.py \
  --map 2x2-brt \
  --resume-from models/MAPPO_colab_2x2_brt \
  --num-iters 40

# 영상 녹화
python record_video_mappo.py \
  --model models/MAPPO_colab_2x2_brt \
  --map 2x2-brt
```
